In [1]:
import numpy as np
from tensorflow.keras import layers, Model, Input
import pandas as pd
import functools

from sklearn.preprocessing import StandardScaler
import scipy
import tensorflow as tf
import matplotlib.pyplot as plt
import os
os.environ["LD_LIBRARY_PATH"] = (
    "/home/asaf/miniconda3/envs/dr2/lib/python3.10/site-packages/nvidia/cudnn/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)
from scipy.stats import norm
from lifelines import CoxPHFitter
from tensorflow.keras.initializers import Constant
from scipy.optimize import minimize
import gc
import numpy as np
import itertools
from tensorflow.keras.optimizers import AdamW, SGD, Adam
import pickle

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Add, Concatenate, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from tensorflow.keras import backend as K
import pickle


2026-05-31 05:46:23.390204: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print(tf.__version__)
tf.config.list_physical_devices('GPU')

2.20.0


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
def simulate(n,sampler,data=None):
    if data == None:
        data = sampler.simulate(n)
    step      = 0.001
    durations = data["durations"]

    t_unique = np.unique(durations)
    t_unique = np.sort(t_unique)
    k = 5  # number of interior points per gap
    midpoints = np.concatenate([
        np.linspace(a, b, num=k+2, dtype=np.float64)[1:-1]  # excludes endpoints a,b
        for a, b in zip(t_unique[:-1], t_unique[1:])
    ]) if len(t_unique) > 1 else np.array([], dtype=np.float64)
    #merged  =  sort + deduplicate
    t_sorted = np.sort(np.unique(np.concatenate([t_unique, midpoints])))
    
    X_covs = data["covs_X"]
    Z_covs = data["covs_Z"]
    E = data["events"]
    T = data["durations"]

    X_list, Z_list, E_list, st_list, et_list, id_list = [], [], [], [], [], []
    
    for i in range(len(X_covs)):
        for j in range(1, len(t_sorted)):
            if t_sorted[j] > T[i]:
                break
            X_list.append(X_covs[i])
            Z_list.append(Z_covs[i])
            E_list.append(E[i] if t_sorted[j] == T[i] else 0)
            st_list.append(t_sorted[j-1])
            et_list.append(t_sorted[j])
            id_list.append(i)
    
    X = np.stack(X_list)
    Z = np.stack(Z_list)
    E = np.array(E_list).reshape(-1, 1)
    st = np.array(st_list).reshape(-1, 1)
    et = np.array(et_list).reshape(-1, 1)
    id_list = np.array(id_list).reshape(-1, 1)
    
    return pd.DataFrame(
        np.concatenate([X, Z, E, st, et, id_list], axis=1),
        columns=[f"X{i}" for i in range(X.shape[1])] +
                [f"Z{i}" for i in range(Z.shape[1])] +
                ["E", "st", "et", "id"]
    ) , data

In [4]:

def loss_beta(beta, Z, g_fixed, et, st, delta):
    beta = beta.reshape(-1, 1)
    linear_term = Z @ beta
    total = g_fixed + linear_term
    integral_term = (et - st) * np.exp(total)
    event_term = delta * total
    return np.sum((integral_term - event_term))

def grad_beta(beta, Z, g_fixed, et, st, delta):
    beta = beta.reshape(-1, 1)
    linear_term = Z @ beta
    total = g_fixed + linear_term
    exp_total = np.exp(total)
    grad = Z * ((et - st) * exp_total - delta)
    return np.sum(grad, axis=0)  # Return 1D array for gradient


def hessp_beta(beta, p, Z, g_fixed, et, st, delta, clip=80.0):
    beta = beta.reshape(-1, 1)
    linear_term = Z @ beta
    total = g_fixed + linear_term
    tot = total
    w = (et - st) * np.exp(np.clip(tot, -clip, clip))   # (n,1)
    p = np.asarray(p).reshape(-1,1)                     # (p,1)
    return (Z.T @ (w * (Z @ p))).ravel()                # (p,)

def loss_beta_sf(beta, Z, g_fixed, et, st, delta):
    x = beta.reshape(-1, 1)
    beta = x[:4]
    shift = x[4]
    scale = x[5]
    linear_term = Z @ beta
    total = g_fixed*scale + shift + linear_term
    integral_term = (et - st) * np.exp(total)
    event_term = delta * total
    return np.sum((integral_term - event_term)) + (tf.reduce_sum(beta**2))*0



In [5]:
def custom_loss(y_true, y_pred, lmbd_cali, lmbd_cor):
    clip_pred  = 30.0
    eps        = 1e-30
    log_switch = 50.0
    dt_tiny    = 1e-8

    et    = y_true[:, 0:1]
    st    = y_true[:, 1:2]
    delta = y_true[:, 2:3]
    train_phase = y_true[:, 3:4]
    r     = y_true[:, 4:]

    train_phase = tf.reduce_mean(train_phase)


    # y_pred is (B,2): [chi, g]
    chi = y_pred[:, 0:1]            # (B,1)
    g   = y_pred[:, 1:2]            # (B,1)  <-- key change

    ne = tf.reduce_sum(delta) + 1e-12
    ne = tf.maximum(ne, 1.0)
    dt = tf.maximum(et - st, 0.0)

    y_safe = tf.clip_by_value(chi, -clip_pred, clip_pred)
    dt_safe = tf.maximum(dt, eps)
    use_log = tf.logical_or(y_safe > log_switch, dt < dt_tiny)

    integral_term = tf.where(
        use_log,
        tf.exp(tf.math.log(dt_safe) + y_safe),
        dt * tf.exp(y_safe)
    )

    event_term = delta * y_safe
    base = tf.reduce_sum(integral_term - event_term) / 150

    # calibration penalty (train-only)
    pen_cali = (ne - tf.reduce_sum(integral_term)) / 150
    pen_cali = lmbd_cali * tf.square(pen_cali)
    pen_cali = pen_cali*train_phase

    # ---- correlation penalty (event-weighted) ----
    w = tf.math.divide_no_nan(delta, ne)        # (B,1)

    # Event-weighted means
    mu_g = tf.reduce_sum(w * g)                 # scalar
    mu_r = tf.reduce_sum(w * r, axis=0)         # (p,)

    # Centered
    g_c = g - mu_g                              # (B,1)
    r_c = r - mu_r[tf.newaxis, :]              # (B,p)

    # Weighted variances
    var_g = tf.reduce_sum(w * tf.square(g_c)) + 1e-12
    var_r = tf.reduce_sum(w * tf.square(r_c), axis=0) + 1e-12  # (p,)

    # Weighted covariances per feature
    cov   = tf.reduce_sum(w * g_c * r_c, axis=0)               # (p,)
    corr2 = tf.square(cov) / (var_g * var_r)                   # (p,)
    
    corr2 = tf.square(corr2)
    
    pen_cor = lmbd_cor * tf.reduce_sum(corr2)
    pen_cor = pen_cor*train_phase

    #tf.print(base,pen_cor)

    return base


In [6]:
def cox_ph_loss_no_ties(y_true, y_pred, clip_pred=80.0, eps=1e-12):
    """
    Negative Cox partial log-likelihood assuming no ties.
    y_true columns: [et, st, delta]; only et and delta are used.
    y_pred: chi = g(et,X) + beta^T Z, shape (batch, 1).
    IMPORTANT: Use full-batch training (risk sets need all samples).
    """
    # flatten to 1D
    T = tf.reshape(y_true[:, 0], [-1])       # observed times (et)
    d = tf.reshape(y_true[:, 1], [-1])       # event indicator (0/1)
    y = tf.reshape(tf.clip_by_value(y_pred[:,0], -clip_pred, clip_pred), [-1])

    # sort by descending time so risk set at i is prefix [:i+1]
    order = tf.argsort(T, direction='DESCENDING')
    y_sorted = tf.gather(y, order)
    d_sorted = tf.gather(d, order)

    # denom_i = log sum_{j<=i} exp(y_j)  (stable enough with clipping)
    cumexp = tf.cumsum(tf.exp(y_sorted))
    log_denom = tf.math.log(tf.maximum(cumexp, eps))

    # partial loglik = sum_{events i} (y_i - log denom_i)
    contrib = d_sorted * (y_sorted - log_denom)
    # negative average over events (scale-stable)
    events = tf.reduce_sum(d_sorted)
    return -tf.reduce_sum(contrib) / tf.maximum(events, 1.0)


# def cox_ph_loss_no_ties(y_true, y_pred, clip_pred=80.0, eps=1e-12):
#     """
#     Negative Cox partial log-likelihood assuming no ties.
#     y_true columns: [et, st, delta]; only et and delta are used.
#     y_pred: chi = g(et,X) + beta^T Z, shape (batch, 1).
#     IMPORTANT: Use full-batch training (risk sets need all samples).
#     """
#     # flatten to 1D
#     T = tf.reshape(y_true[:, 0], [-1])       # observed times (et)
#     d = tf.reshape(y_true[:, 1], [-1])       # event indicator (0/1)
#     #y = tf.reshape(tf.clip_by_value(y_pred[:,0], -clip_pred, clip_pred), [-1])
#     y = y_pred[:, 0]            # (B,1)
#     g   = y_pred[:, 1]            # (B,1)  <-- key change

#     y = y -tf.reduce_mean(g)
#     # sort by descending time so risk set at i is prefix [:i+1]
#     order = tf.argsort(T, direction='DESCENDING')
#     y_sorted = tf.gather(y, order)
#     d_sorted = tf.gather(d, order)

#     # denom_i = log sum_{j<=i} exp(y_j)  (stable enough with clipping)
#     cumexp = tf.cumsum(tf.exp(y_sorted))
#     log_denom = tf.math.log(tf.maximum(cumexp, eps))

#     # partial loglik = sum_{events i} (y_i - log denom_i)
#     contrib = d_sorted * (y_sorted - log_denom)
#     # negative average over events (scale-stable)
#     events = tf.reduce_sum(d_sorted)
#     #pen = f.reduce_mean(g)
#     return -tf.reduce_sum(contrib)

In [7]:
class L1Var(tf.keras.regularizers.Regularizer):
    def __init__(self, coeff_var): self.c = coeff_var
    def __call__(self, x): return self.c * tf.reduce_sum(tf.abs(x))

        
def build_model(input_dim_X, input_dim_Z, nn_config, beta_init,lmbd_l1=0):
    X_input  = Input(shape=(input_dim_X,), name="X_input")
    Z_input  = Input(shape=(input_dim_Z,), name="Z_input")
    et_input = Input(shape=(1,),           name="et_input")    # (time) used in g(t,X)


    body_layers = []
    head_layers = []
    reg = L1Var(lmbd_l1)
    g_input = Concatenate(name="concat_xt")([et_input, X_input])
    x = g_input
    for _ in range(nn_config["n_hidden_layers"]):
        tmp = Dense(nn_config["hidden_layers_nodes"], activation='relu',kernel_regularizer=reg,bias_regularizer=reg)
        body_layers += [tmp]
        x = tmp(x)
        x = tf.keras.layers.Activation('relu')(x)

    tmp = Dense(1, activation=None, name="g_out",kernel_regularizer=reg,bias_regularizer=None)
    g_out = tmp(x)
    body_layers += [tmp]
    beta_out = Dense(1, use_bias=False, activation=None, name="beta_layer",kernel_initializer=Constant(beta_init))(Z_input)
    head_layers = [beta_out]
    out = Add(name="chi")([g_out, beta_out])
    chi_g = Concatenate(axis=1, name="chi_g")([out, g_out])


    model = Model(inputs=[X_input, Z_input, et_input], outputs=chi_g)
    return model, body_layers, head_layers, reg

def compile_cox_ph(model,nn_config,optimizer=None,lmbd_cali=0,lmbd_cor=0):
    if optimizer == None:
        optimizer = Adam(learning_rate=nn_config["learning_rate"])
    model.compile(optimizer=optimizer, loss=cox_ph_loss_no_ties, jit_compile=False)
    return model,optimizer


def compile_baseliness(model,nn_config,optimizer=None,lmbd_cali=0,lmbd_cor=0):
    if optimizer == None:
        optimizer = Adam(learning_rate=nn_config["learning_rate"])

    def inner_loss(y_true, y_pred):
        return custom_loss(y_true, y_pred,lmbd_cali=lmbd_cali,lmbd_cor=lmbd_cor)
    model.compile(optimizer=optimizer, loss=inner_loss , jit_compile=False)
    return model, optimizer


def build_mse(input_dim_X, nn_config):
    X_input = Input(shape=(input_dim_X,), name="X_input")
    et_input = Input(shape=(1,), name="et_input")  # Time t used in g(t, X)

    # g(t, X) modeled by feeding [t, X]
    g_input = Concatenate()([et_input, X_input])
    x = g_input
    for _ in range(nn_config["n_hidden_layers"]):
        x = Dense(nn_config["hidden_layers_nodes"], activation=nn_config["activation"])(x)
        x = Dropout(nn_config["dropout"])(x)
    g_out = Dense(1, activation='linear')(x)

    model = Model(inputs=[X_input, et_input], outputs=g_out)

    optimizer = tf.keras.optimizers.get(nn_config["optimizer"])
    optimizer.learning_rate = nn_config["learning_rate"]

    model.compile(optimizer=optimizer, loss="mse", jit_compile=False)
    return model



In [8]:
def get_r2(t,p):
    tot = ((t-np.mean(t))**2).sum()
    cond = ((t-p)**2).sum()
    return 1- (cond/tot)

n = 4000
n_jobs = 2
N_BOOTSTRAPS = 1
TOP_K = 1
val_frac = 0.33
TRUE_THETA = np.array([2,-1])
max_attempts = 5
q = 0.0
lp = 0


nn_config = {
    "hidden_layers_nodes": 40,
    "n_hidden_layers": 5,
    "learning_rate": 0.001,
    "activation": 'relu', 
    "optimizer": 'adam',
    "batch_size": 128,
    "patience": 50,
    "dropout": 0.0,

    "lmbd_L1" : 1e-3,
    "lmbd_cali": 0,
    "lmbd_cor": 0,
}
n = 4000
p = 2
in_ci_95 = np.zeros(p)
in_ci_90 = np.zeros(p)

in_ci_95_opt = np.zeros(p)
in_ci_90_opt = np.zeros(p)

beta_all = []
beta_all_opt = []
cov_all = []
val_loss_all = []
train_loss_all = []
r2_all = []

deep_cox_result = []
cox_result = []

In [9]:
from sklearn.model_selection import GroupKFold
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

def crossfit_event_residuals(df, nn_config, K=5, time_scale=30.0):
    """
    OOF residuals at EVENTS ONLY for correlation penalty.
    Trains each r_j on train-fold EVENTS and predicts for held-out fold EVENTS.
    Returns r_oof with shape (n, p), filled on events (delta==1), zeros elsewhere.
    """

    X_cols = [c for c in df.columns if c.startswith("X")]
    Z_cols = [c for c in df.columns if c.startswith("Z")]
    ids = df["id"]
    
    X = df[X_cols].values
    Z = df[Z_cols].values
    et = df["et"].values.reshape(-1, 1)
    st = df["st"].values.reshape(-1, 1)
    E = df["E"].values.reshape(-1, 1)

    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    et = np.asarray(et, dtype=np.float32).reshape(-1, 1)
    E = np.asarray(E, dtype=np.int32).reshape(-1)
    ids = np.asarray(ids)

    n, p = Z.shape
    r_oof = np.zeros((n, p), dtype=np.float32)

    # split by subject to avoid leakage across the same id
    gkf = GroupKFold(n_splits=min(K, np.unique(ids).size))
    for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, groups=ids)):
        tr_ev = tr_idx[E[tr_idx] == 1]
        te_ev = te_idx[E[te_idx] == 1]
        if tr_ev.size == 0 or te_ev.size == 0:
            continue  # skip degenerate fold

        for j in range(p):
            rmodel = build_mse(input_dim_X=X.shape[1], nn_config=nn_config)
            es = EarlyStopping(monitor='val_loss', patience=nn_config["patience"],
                               restore_best_weights=True)
            # fit only on EVENTS in the training folds
            rmodel.fit(
                [X[tr_ev], et[tr_ev] / time_scale], Z[tr_ev, j],
                validation_data=([X[te_ev], et[te_ev] / time_scale], Z[te_ev, j]),
                batch_size=nn_config["batch_size"],
                epochs=5000, callbacks=[es], verbose=0
            )
            # predict on EVENTS in the held-out fold
            pred_te_ev = rmodel.predict([X[te_ev], et[te_ev] / time_scale], verbose=0,batch_size=10000).reshape(-1)
            r_oof[te_ev, j] = Z[te_ev, j] - pred_te_ev
        tf.keras.backend.clear_session()
        gc.collect()   
    # non-events remain zero; OK because penalty uses event weights
    return r_oof


In [10]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # hide INFO/WARNING from TF C++ logs
# Optional: tone down XLA autotuner chatter / overhead
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=2"  # or "0" to disable autotuning

In [11]:
import warnings
warnings.filterwarnings("ignore")


In [12]:
df = pd.read_csv("rot.csv")

In [13]:
df["er"].median()

np.float64(61.0)

In [14]:
#df = df[df.er >= 61]

In [15]:
df["low_er"] = df.er < 61

In [16]:
df["age"] = (df["age"] - df["age"].mean())/df["age"].std()

In [17]:
df["pgr"] = (df["pgr"] - df["pgr"].mean())/df["pgr"].std()
df["er"] = (df["er"] - df["er"].mean())/df["er"].std()

In [18]:
df["age"]

0       1.462338
1       1.848350
2      -0.853730
3       1.153529
4       1.539541
          ...   
2977    1.385136
2978    1.539541
2979    1.848350
2980    1.230732
2981    0.999125
Name: age, Length: 2982, dtype: float64

In [19]:
df["size_fac"] = df["size"].astype("category").cat.codes

In [20]:
df["hormon_low"] = (df["hormon"]*df["low_er"])
df["hormon_high"] = (df["hormon"]*(1-df["low_er"]))

df["chemo_low"] = (df["chemo"]*df["low_er"])
df["chemo_high"] = (df["chemo"]*(1-df["low_er"]))

In [21]:
data = {
    "covs_X" : df[["age","meno","size_fac","grade","nodes","pgr","er"]].values,
    "covs_Z" : df[["hormon_low","hormon_high","chemo_low","chemo_high"]].values,
    "events" : np.max([df["recur"],df["death"]],axis=0),
    "durations" : np.min([df["rtime"],df["dtime"]],axis=0),
}


In [22]:
data["durations"] = data["durations"]/data["durations"].max()

In [23]:
df_all, data = simulate(n,None,data)
X_cols = [c for c in df_all.columns if c.startswith("X")]
Z_cols = [c for c in df_all.columns if c.startswith("Z")]
ids = df_all["id"].unique()

p = len(Z_cols)

navie_cox_df = np.concatenate([data["covs_X"],data["covs_Z"],data["durations"].reshape(-1,1),data["events"].reshape(-1,1)],axis=1)
navie_cox_df = pd.DataFrame(navie_cox_df,columns=X_cols + Z_cols + ["durations","events"])


cph = CoxPHFitter()
cph.fit(navie_cox_df, duration_col="durations", event_col="events", show_progress=False)

<lifelines.CoxPHFitter: fitted with 2982 total observations, 1269 right-censored observations>

In [24]:
coef_series = cph.params_

In [26]:
coef_series[-p:]

covariate
Z0    0.115400
Z1   -0.289921
Z2   -0.022702
Z3   -0.088301
Name: coef, dtype: float64

In [30]:
np.diag(cph.variance_matrix_)[-p:]

array([0.01121944, 0.01086173, 0.00676878, 0.00965675])

In [23]:
#time_scale = 30
df_all, data = simulate(n,None,data)
X_cols = [c for c in df_all.columns if c.startswith("X")]
Z_cols = [c for c in df_all.columns if c.startswith("Z")]
ids = df_all["id"].unique()

p = len(Z_cols)
r_oof = crossfit_event_residuals(df_all,nn_config)


I0000 00:00:1779786940.169934    1658 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20846 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:35:00.0, compute capability: 8.9


In [24]:
nn_config = {
    "hidden_layers_nodes": 40,
    "n_hidden_layers": 5,
    "learning_rate": 0.001,
    "activation": 'relu', 
    "optimizer": 'adam',
    "batch_size": 128,
    "patience": 50,
    "dropout": 0.0,

    "lmbd_L1" : 3e-4,
    "lmbd_cali": 0,
    "lmbd_cor": 0,
}

In [ ]:
time_scale=1
bets = []
gkf = GroupKFold(n_splits=5,shuffle=True,random_state=13)
groups = df_all["id"].values

fold_data = []  # collect per-fold inputs/targets and anything else you want

for fold_idx, (tr_rows, va_rows) in enumerate(gkf.split(df_all, groups=groups), start=1):
    best_beta = None
    best_val = np.inf
    best_conif = None
    # Row-wise splits that respect IDs
    df_train = df_all.iloc[tr_rows].copy()
    df_val   = df_all.iloc[va_rows].copy()
    
    r_mat_train = r_oof[tr_rows]
    r_mat_val = r_oof[va_rows]
    
    
    # Remove times untill first event
    t_l_bound = df_train[df_train.E == 1].et.min()
    t_h_bound = df_train[df_train.E == 1].et.max()
    
    r_mat_train = r_mat_train[df_train.st >= t_l_bound]
    r_mat_val = r_mat_val[df_val.st >= t_l_bound]
    
    df_train = df_train[df_train.st >= t_l_bound]
    df_val = df_val[df_val.st >= t_l_bound]
    
    navie_cox_df = np.concatenate([data["covs_X"],data["covs_Z"],data["durations"].reshape(-1,1),data["events"].reshape(-1,1)],axis=1)
    navie_cox_df = pd.DataFrame(navie_cox_df,columns=X_cols + Z_cols + ["durations","events"])
    
    
    cph = CoxPHFitter()
    cph.fit(navie_cox_df, duration_col="durations", event_col="events", show_progress=False)
    
    
    # Extract beta coefficients for Z variables
    coef_series = cph.params_
    beta_init = coef_series[Z_cols].values.reshape(-1, 1)
    print("β_hat cox:", beta_init)
    
    
    X_train = df_train[X_cols].values
    Z_train = df_train[Z_cols].values
    et_train = df_train["et"].values.reshape(-1, 1) / time_scale
    st_train = df_train["st"].values.reshape(-1, 1) / time_scale
    E_train = df_train["E"].values.reshape(-1, 1)
    
    
    X_val = df_val[X_cols].values
    Z_val = df_val[Z_cols].values
    et_val = df_val["et"].values.reshape(-1, 1) / time_scale
    st_val = df_val["st"].values.reshape(-1, 1) / time_scale
    E_val = df_val["E"].values.reshape(-1, 1)
    
    mask_tr = (E_train.reshape(-1,) == 1)
    mask_va = (E_val.reshape(-1,) == 1)
    
    r_mean_tr = r_mat_train[mask_tr].mean(axis=0, keepdims=True)
    r_mat_train[mask_tr] -= r_mean_tr
    
    r_mean_va = r_mat_val[mask_va].mean(axis=0, keepdims=True)
    r_mat_val[mask_va] -= r_mean_va
    # then use these in your inputs as you already do:
    train_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_train, Z_train, et_train, r_mat_train, E_train
    ])
    val_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_val,   Z_val,   et_val,   r_mat_val,   E_val
    ])
    
    y_train = np.concatenate([et_train,
                              st_train,
                              E_train,
                              np.ones(et_train.shape),
                              r_mat_train], axis=1) 
    
    y_val = np.concatenate([et_val,
                            st_val,
                            E_val,
                            np.zeros(et_val.shape),
                            r_mat_val], axis=1)
    
    train_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_train,
        Z_train,
        et_train,
    ])
    
    val_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_val,
        Z_val,
        et_val,
    ])
    
    print(f"\n=== Fold {fold_idx} ===")
    for n_layer in [2,3,4]:
        for n_nodes in [20,40,80]:
            for lmbd_l1 in [0,1e-4]:

                nn_config = {
                    "hidden_layers_nodes": n_nodes,
                    "n_hidden_layers": n_layer,
                    "learning_rate": 0.001,
                    "activation": 'relu', 
                    "optimizer": 'adam',
                    "batch_size": 128,
                    "patience": 50,
                    "dropout": 0.0,
                
                    "lmbd_L1" : lmbd_l1,
                    "lmbd_cali": 0,
                    "lmbd_cor": 0,
                }
        
                optimizer = Adam(learning_rate=0.001)    
                model,body,head,reg = build_model(input_dim_X=X_train.shape[1],
                                    input_dim_Z=Z_train.shape[1],
                                    nn_config=nn_config,
                                    beta_init=beta_init,lmbd_l1=nn_config["lmbd_L1"])
                
                model, optimizer = compile_baseliness(model,nn_config,optimizer=optimizer,lmbd_cali=nn_config["lmbd_cali"],lmbd_cor=nn_config["lmbd_cor"])
                beta_hat = model.get_layer("beta_layer").get_weights()[0].reshape((p,))
                g_train = model.predict(train_inp, batch_size=100000,verbose=0)[:,1].reshape(-1,1)
                x0 = np.concat((beta_hat,[0,1]))
                res = minimize(
                    fun=loss_beta_sf,
                    x0=x0,
                    args=(Z_train.astype(np.float64), g_train.astype(np.float64),
                          et_train.astype(np.float64), st_train.astype(np.float64),
                          E_train.astype(np.float64)),
                    method="L-BFGS-B",
                )
                res.x
                beta_hat = res.x[:p]
                shift = res.x[p]
                scale = res.x[p + 1]
                
                
                dense = model.get_layer("g_out")  # Dense(1)
                W, b = dense.get_weights()        # W shape (1,1), b shape (1,)
                dense.set_weights([W * scale, b* scale + shift])
                
                optimizer.learning_rate = 0.001
                es = EarlyStopping(monitor='val_loss', patience=35,
                                   restore_best_weights=True)
                history = model.fit(
                    train_inp, y_train,
                    validation_data=(val_inp, y_val),
                    batch_size=int(X_train.shape[0]/20),
                    epochs=3000,
                    callbacks=[es],
                    verbose = 0
                )
                
                beta_hat = model.get_layer("beta_layer").get_weights()[0].reshape((p,))
#                print(beta_hat)
                g_train = model.predict(train_inp, batch_size=10000,verbose=0)[:,1].reshape(-1,1)
                res = minimize(
                    loss_beta,
                    x0=beta_hat,
                    args=(Z_train, g_train, et_train, st_train, E_train),
                    method='L-BFGS-B',
                    jac=grad_beta
                )
                beta_hat = res.x
                print(beta_hat)
                print("!!!!!")
                g_val = model.predict(val_inp, batch_size=10000,verbose=0)[:,1].reshape(-1,1)
                ls = loss_beta(beta_hat,Z_val, g_val, et_val, st_val, E_val)
                if  ls < best_val:
                    best_val = ls
                    best_beta = beta_hat
                    best_conif = nn_config.copy()
                
    bets += [best_beta]
                
    print("###################")
    

β_hat cox: [[ 0.1153995 ]
 [-0.28992057]
 [-0.02270232]
 [-0.08830107]]

=== Fold 1 ===


In [33]:
beta_hat

array([ 0.1153995 , -0.28992057, -0.02270233, -0.08830107], dtype=float32)

In [36]:
bets

[array([-0.34131396, -0.62107389, -0.36641821, -0.50591568]),
 array([-0.2207472 , -0.55785394, -0.34063484, -0.3566298 ]),
 array([-0.22875661, -0.46131492, -0.33921245, -0.34116298]),
 array([-0.11636376, -0.44474865, -0.30185005, -0.29264787]),
 array([-0.32024621, -0.51918931, -0.39489928, -0.33511854])]

In [ ]:
g_train.max(),g_train.min()

In [ ]:
pickle.dump([bets,best_conif],open("res_rot_csv,pkl","wb"))

In [37]:
best_beta

array([-0.32024621, -0.51918931, -0.39489928, -0.33511854])

In [26]:
best_conif

{'hidden_layers_nodes': 20,
 'n_hidden_layers': 2,
 'learning_rate': 0.001,
 'activation': 'relu',
 'optimizer': 'adam',
 'batch_size': 128,
 'patience': 50,
 'dropout': 0.0,
 'lmbd_L1': 0,
 'lmbd_cali': 0,
 'lmbd_cor': 0}

In [ ]:

bets = []


all_bets_all = []
all_scores_all = []
fold_data = []  # collect per-fold inputs/targets and anything else you want


gkf = GroupKFold(n_splits=5,shuffle=True)
groups = df_all["id"].values
split = []
for fold_idx, (tr_rows, va_rows) in enumerate(gkf.split(df_all, groups=groups), start=1):
    split += [[fold_idx,(tr_rows,va_rows)]]

for fold_idx, (tr_rows, va_rows) in split:
    
    best_beta = None
    best_val = np.inf
    best_conif = None
    all_bets = []
    all_scores = []
    # Row-wise splits that respect IDs
    df_train = df_all.iloc[tr_rows].copy()
    df_val   = df_all.iloc[va_rows].copy()
    
    r_mat_train = r_oof[tr_rows]
    r_mat_val = r_oof[va_rows]
    
    
    # Remove times untill first event
    t_l_bound = df_train[df_train.E == 1].et.min()
    t_h_bound = df_train[df_train.E == 1].et.max()
    
    r_mat_train = r_mat_train[df_train.st >= t_l_bound]
    r_mat_val = r_mat_val[df_val.st >= t_l_bound]
    
    df_train = df_train[df_train.st >= t_l_bound]
    df_val = df_val[df_val.st >= t_l_bound]
    
    navie_cox_df = np.concatenate([data["covs_X"],data["covs_Z"],data["durations"].reshape(-1,1),data["events"].reshape(-1,1)],axis=1)
    navie_cox_df = pd.DataFrame(navie_cox_df,columns=X_cols + Z_cols + ["durations","events"])
    
    
    cph = CoxPHFitter()
    cph.fit(navie_cox_df, duration_col="durations", event_col="events", show_progress=False)
    
    
    # Extract beta coefficients for Z variables
    coef_series = cph.params_
    beta_init = coef_series[Z_cols].values.reshape(-1, 1)
    print("β_hat cox:", beta_init)
    
    
    X_train = df_train[X_cols].values
    Z_train = df_train[Z_cols].values
    et_train = df_train["et"].values.reshape(-1, 1) / time_scale
    st_train = df_train["st"].values.reshape(-1, 1) / time_scale
    E_train = df_train["E"].values.reshape(-1, 1)
    
    
    X_val = df_val[X_cols].values
    Z_val = df_val[Z_cols].values
    et_val = df_val["et"].values.reshape(-1, 1) / time_scale
    st_val = df_val["st"].values.reshape(-1, 1) / time_scale
    E_val = df_val["E"].values.reshape(-1, 1)
    
    mask_tr = (E_train.reshape(-1,) == 1)
    mask_va = (E_val.reshape(-1,) == 1)
    
    r_mean_tr = r_mat_train[mask_tr].mean(axis=0, keepdims=True)
    r_mat_train[mask_tr] -= r_mean_tr
    
    r_mean_va = r_mat_val[mask_va].mean(axis=0, keepdims=True)
    r_mat_val[mask_va] -= r_mean_va
    # then use these in your inputs as you already do:
    train_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_train, Z_train, et_train, r_mat_train, E_train
    ])
    val_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_val,   Z_val,   et_val,   r_mat_val,   E_val
    ])
    
    y_train = np.concatenate([et_train,
                              st_train,
                              E_train,
                              np.ones(et_train.shape),
                              r_mat_train], axis=1) 
    
    y_val = np.concatenate([et_val,
                            st_val,
                            E_val,
                            np.zeros(et_val.shape),
                            r_mat_val], axis=1)
    
    train_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_train,
        Z_train,
        et_train,
    ])
    
    val_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_val,
        Z_val,
        et_val,
    ])
    for i in range(15):

            nn_config = {
                "hidden_layers_nodes": 20,
                "n_hidden_layers": 2,
                "learning_rate": 0.0001,
                "activation": 'relu', 
                "optimizer": 'adam',
                "batch_size": 128,
                "patience": 50,
                "dropout": 0.0,
            
                "lmbd_L1" : 0,
                "lmbd_cali": 0,
                "lmbd_cor": 0,
            }
    
            optimizer = Adam(learning_rate=0.001,clipnorm=1)    
            model,body,head,reg = build_model(input_dim_X=X_train.shape[1],
                                input_dim_Z=Z_train.shape[1],
                                nn_config=nn_config,
                                beta_init=beta_init,lmbd_l1=nn_config["lmbd_L1"])
            
            model, optimizer = compile_baseliness(model,nn_config,optimizer=optimizer,lmbd_cali=nn_config["lmbd_cali"],lmbd_cor=nn_config["lmbd_cor"])
            beta_hat = model.get_layer("beta_layer").get_weights()[0].reshape((p,))
            g_train = model.predict(train_inp, batch_size=100000,verbose=0)[:,1].reshape(-1,1)
            x0 = np.concat((beta_hat,[0,1]))
            res = minimize(
                fun=loss_beta_sf,
                x0=x0,
                args=(Z_train.astype(np.float64), g_train.astype(np.float64),
                      et_train.astype(np.float64), st_train.astype(np.float64),
                      E_train.astype(np.float64)),
                method="L-BFGS-B",
            )
            res.x
            beta_hat = res.x[:2]
            shift = res.x[2]
            scale = res.x[3]
            
            
            dense = model.get_layer("g_out")  # Dense(1)
            W, b = dense.get_weights()        # W shape (1,1), b shape (1,)
            dense.set_weights([W * scale, b* scale + shift])
            
            optimizer.learning_rate = 0.0001
            es = EarlyStopping(monitor='val_loss', patience=35,
                               restore_best_weights=True)
            history = model.fit(
                train_inp, y_train,
                validation_data=(val_inp, y_val),
                batch_size=int(X_train.shape[0]/20),
                epochs=3000,
                callbacks=[es],
                verbose = 0
            )
            
            beta_hat = model.get_layer("beta_layer").get_weights()[0].reshape((p,))
#                print(beta_hat)
            g_train = model.predict(train_inp, batch_size=10000,verbose=0)[:,1].reshape(-1,1)
            res = minimize(
                loss_beta,
                x0=beta_hat,
                args=(Z_train, g_train, et_train, st_train, E_train),
                method='L-BFGS-B',
                jac=grad_beta
            )
            beta_hat = res.x
            print(beta_hat)
            print("!!!!!")
            g_val = model.predict(val_inp, batch_size=10000,verbose=0)[:,1].reshape(-1,1)
            ls = loss_beta(beta_hat,Z_val, g_val, et_val, st_val, E_val)

            all_bets += [beta_hat]
            all_scores += [ls]
            if  ls < best_val:
                best_val = ls
                best_beta = beta_hat
                best_conif = nn_config.copy()
            
    bets += [best_beta]
    all_bets_all += [all_bets]
    all_scores_all += [all_scores]
    print("###################")

β_hat cox: [[ 0.1153995 ]
 [-0.28992057]
 [-0.02270232]
 [-0.08830107]]


(14886120, 7)

In [29]:
pickle.dump((all_bets_all,all_scores_all,split,bets),  open("rot_res_15.pkl","wb"))

In [29]:
k = 5
for i in range(5):
    aa = np.stack(all_bets_all[i])
    bb = all_scores_all[i]
    print(np.nanmean(aa[np.argsort(bb)[:k],:],axis=0))

[-0.21391453 -0.5439394  -0.2965149  -0.32229394]
[-0.13782817 -0.54038482 -0.27828678 -0.21509517]
[-0.19389883 -0.47549385 -0.28466344 -0.3799165 ]
[-0.16518725 -0.47693267 -0.32140218 -0.36564746]
[-0.32203298 -0.5067167  -0.35403821 -0.20231802]


In [30]:
bets

[array([-0.17839563, -0.54934747, -0.21479223, -0.30151304]),
 array([-0.17580494, -0.58410153, -0.32802173, -0.30864412]),
 array([-0.23525769, -0.47699443, -0.40103834, -0.39285953]),
 array([-0.18672454, -0.49708808, -0.38841601, -0.40734889]),
 array([-0.29269974, -0.4791556 , -0.33246685, -0.1940507 ])]

In [31]:
np.diag(np.linalg.inv(r_mat_train.T @ r_mat_train))**0.5

array([0.13279645, 0.12208722, 0.13605987, 0.14318873], dtype=float32)

In [32]:
def cox_ph_loss_no_ties(y_true, y_pred, clip_pred=80.0, eps=1e-12):
    """
    Negative Cox partial log-likelihood assuming no ties.
    y_true columns: [et, st, delta]; only et and delta are used.
    y_pred: chi = g(et,X) + beta^T Z, shape (batch, 1).
    IMPORTANT: Use full-batch training (risk sets need all samples).
    """
    # flatten to 1D
    T = tf.reshape(y_true[:, 0], [-1])       # observed times (et)
    d = tf.reshape(y_true[:, 2], [-1])       # event indicator (0/1)
    y = tf.reshape(tf.clip_by_value(y_pred[:,0], -clip_pred, clip_pred), [-1])

    # sort by descending time so risk set at i is prefix [:i+1]
    order = tf.argsort(T, direction='DESCENDING')
    y_sorted = tf.gather(y, order)
    d_sorted = tf.gather(d, order)

    # denom_i = log sum_{j<=i} exp(y_j)  (stable enough with clipping)
    cumexp = tf.cumsum(tf.exp(y_sorted))
    log_denom = tf.math.log(tf.maximum(cumexp, eps))

    # partial loglik = sum_{events i} (y_i - log denom_i)
    contrib = d_sorted * (y_sorted - log_denom)
    # negative average over events (scale-stable)
    events = tf.reduce_sum(d_sorted)
    return -tf.reduce_sum(contrib) / tf.maximum(events, 1.0)

In [33]:
def create_ph_df(data):
    sz = data["events"].shape[0]
    ret = np.concatenate([data["covs_X"],
                    data["covs_Z"],
                    data["events"].reshape(-1,1),
                    np.zeros((sz,1)),
                    data["durations"].reshape(-1,1),
                    np.array(range(0,sz)).reshape(-1,1)],axis=1)
    cols = [f"X{i}" for i in range(data["covs_X"].shape[1])]
    cols += [f"Z{i}" for i in range(data["covs_Z"].shape[1])]
    cols += ["E","st","et","id"]
    return pd.DataFrame(ret,columns=cols)

In [34]:
import numpy as np

def loss_beta_cox(beta, Z, g_fixed, et, st, delta):
    """
    Negative partial log-likelihood (Breslow ties) with per-sample offset g_fixed.
    Risk set R(t_i) = {j : et_j >= et_i}. Ignores st.
    """
    beta  = np.asarray(beta).reshape(-1, 1)
    Z     = np.asarray(Z, dtype=float)
    g     = np.asarray(g_fixed, dtype=float).reshape(-1)
    t     = np.asarray(et, dtype=float).reshape(-1)
    d     = np.asarray(delta, dtype=float).astype(bool).reshape(-1)

    if not d.any():
        return 0.0

    lp = (Z @ beta).ravel() + g
    w  = np.exp(np.clip(lp, -50.0, 50.0))

    # sort by time DESC so prefixes are risk sets
    order = np.argsort(-t)
    t_s, w_s = t[order], w[order]

    W_cum = np.cumsum(w_s)
    # last index within each equal-time block
    last = np.r_[t_s[1:] < t_s[:-1], True]
    times_u = t_s[last]
    denom_u = W_cum[last]
    denom_map = dict(zip(times_u, denom_u))

    # group events by time
    t_ev = t[d]
    lp_ev = lp[d]
    ev_ord = np.argsort(t_ev)
    t_ev, lp_ev = t_ev[ev_ord], lp_ev[ev_ord]
    starts = np.r_[0, np.flatnonzero(t_ev[1:] != t_ev[:-1]) + 1]
    ends   = np.r_[starts[1:], len(t_ev)]

    ll = 0.0
    eps = 1e-300
    for s, e in zip(starts, ends):
        ti   = t_ev[s]
        d_t  = e - s
        ll  += lp_ev[s:e].sum() - d_t * np.log(denom_map.get(ti, 0.0) + eps)

    return -float(ll)


def grad_beta_cox(beta, Z, g_fixed, et, st, delta):
    """
    Gradient of negative partial log-likelihood (Breslow) w.r.t. beta.
    Ignores st. Returns a 1D array.
    """
    beta  = np.asarray(beta).reshape(-1, 1)
    Z     = np.asarray(Z, dtype=float)
    g     = np.asarray(g_fixed, dtype=float).reshape(-1)
    t     = np.asarray(et, dtype=float).reshape(-1)
    d     = np.asarray(delta, dtype=float).astype(bool).reshape(-1)

    if not d.any():
        return np.zeros(Z.shape[1], dtype=float)

    lp = (Z @ beta).ravel() + g
    w  = np.exp(np.clip(lp, -50.0, 50.0))

    order = np.argsort(-t)
    t_s, w_s, Z_s = t[order], w[order], Z[order, :]

    W_cum = np.cumsum(w_s)
    U_cum = np.cumsum(w_s[:, None] * Z_s, axis=0)

    last = np.r_[t_s[1:] < t_s[:-1], True]
    times_u = t_s[last]
    denom_u = W_cum[last]
    numZ_u  = U_cum[last, :]

    eps = 1e-300
    EZ_u = numZ_u / (denom_u[:, None] + eps)  # E[Z | risk at time]

    # quick lookup by time
    EZ_map = {ti: ez for ti, ez in zip(times_u, EZ_u)}

    # group events by time
    t_ev = t[d]
    Z_ev = Z[d, :]
    ev_ord = np.argsort(t_ev)
    t_ev, Z_ev = t_ev[ev_ord], Z_ev[ev_ord, :]
    starts = np.r_[0, np.flatnonzero(t_ev[1:] != t_ev[:-1]) + 1]
    ends   = np.r_[starts[1:], len(t_ev)]

    grad = np.zeros(Z.shape[1], dtype=float)
    for s, e in zip(starts, ends):
        ti  = t_ev[s]
        d_t = e - s
        grad += Z_ev[s:e, :].sum(axis=0) - d_t * EZ_map[ti]

    return -grad


In [36]:
train_ids_fold = []
val_ids_fold = []
df_all, data = simulate(n,None,data)
for fold_idx, (tr_rows, va_rows) in split[:5]:
    train_ids_fold += [df_all.iloc[tr_rows]["id"].apply(int).unique()]
    val_ids_fold += [df_all.iloc[va_rows]["id"].apply(int).unique()]

In [38]:
df_all = create_ph_df(data)
X_cols = [c for c in df_all.columns if c.startswith("X")]
Z_cols = [c for c in df_all.columns if c.startswith("Z")]
r_oof = np.zeros((df_all.shape[0],len(Z_cols)))

In [39]:

bets_cox = []


all_bets_all_cox = []
all_scores_all_cox = []
fold_data_cox = []  # collect per-fold inputs/targets and anything else you want

for train_ids in train_ids_fold:
    
    best_beta = None
    best_val = np.inf
    best_conif = None
    all_bets = []
    all_scores = []
    # Row-wise splits that respect IDs
    df_train = df_all[df_all["id"].isin(train_ids)]
    df_val = df_all[~df_all["id"].isin(train_ids)]
    
    r_mat_train = r_oof[df_all["id"].isin(train_ids)]
    r_mat_val = r_oof[~df_all["id"].isin(train_ids)]

    navie_cox_df = np.concatenate([data["covs_X"],data["covs_Z"],data["durations"].reshape(-1,1),data["events"].reshape(-1,1)],axis=1)
    navie_cox_df = pd.DataFrame(navie_cox_df,columns=X_cols + Z_cols + ["durations","events"])
        
    cph = CoxPHFitter()
    cph.fit(navie_cox_df, duration_col="durations", event_col="events", show_progress=False)

    
    # Extract beta coefficients for Z variables
    coef_series = cph.params_
    beta_init = coef_series[Z_cols].values.reshape(-1, 1)
    cox_result += [beta_init]
    print("β_hat cox:", beta_init)

    
    X_train = df_train[X_cols].values
    Z_train = df_train[Z_cols].values
    et_train = df_train["et"].values.reshape(-1, 1)
    st_train = df_train["st"].values.reshape(-1, 1)
    E_train = df_train["E"].values.reshape(-1, 1)


    X_val = df_val[X_cols].values
    Z_val = df_val[Z_cols].values
    et_val = df_val["et"].values.reshape(-1, 1)
    st_val = df_val["st"].values.reshape(-1, 1)
    E_val = df_val["E"].values.reshape(-1, 1)

    mask_tr = (E_train.reshape(-1,) == 1)
    mask_va = (E_val.reshape(-1,) == 1)
    
    # then use these in your inputs as you already do:
    train_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_train, Z_train, np.zeros(E_train.shape)
    ])
    val_inp = tuple(np.asarray(arr, dtype=np.float32) for arr in [
        X_val,   Z_val,   np.zeros(E_val.shape)
    ])

    y_train = np.concatenate([et_train,
                              st_train,
                              E_train,
                              np.ones(et_train.shape),
                              r_mat_train], axis=1) 

    y_val = np.concatenate([et_val,
                            st_val,
                            E_val,
                            np.zeros(et_val.shape),
                            r_mat_val], axis=1)

    for i in range(15):

            nn_config = {
                "hidden_layers_nodes": 64,
                "n_hidden_layers": 3,
                "learning_rate": 0.001,
                "activation": 'relu', 
                "optimizer": 'adam',
                "batch_size": 128,
                "patience": 50,
                "dropout": 0.0,
            
                "lmbd_L1" : 0,
                "lmbd_cali": 0,
                "lmbd_cor": 0,
            }
    
            optimizer = Adam(learning_rate=0.0001)   
            model,body,head,reg = build_model(input_dim_X=X_train.shape[1],
                                input_dim_Z=Z_train.shape[1],
                                nn_config=nn_config,
                                beta_init=beta_init,lmbd_l1=nn_config["lmbd_L1"])
            
            model, optimizer = compile_cox_ph(model,nn_config,optimizer=optimizer)
            early_stop = EarlyStopping(
                monitor='val_loss',
                patience=150,
                restore_best_weights=True
            )       
            history = model.fit(
                train_inp, y_train,
                validation_data=(val_inp, y_val),
                batch_size=int(X_train.shape[0]/20),
                epochs=50000,
                callbacks=[early_stop],
                verbose = 0
            )
            beta_hat = model.get_layer("beta_layer").get_weights()[0].reshape(p,)
            print(beta_hat)

            g_train = model.predict(train_inp, batch_size=10000,verbose=0)[:,1].reshape(-1,1)
            res = minimize(
                loss_beta_cox,
                x0=beta_hat,
                args=(Z_train, g_train, et_train, st_train, E_train),
                method='L-BFGS-B',
                #jac=grad_beta
            )
            beta_hat = res.x
            print(beta_hat)
            print("!!!!!")
        
            g_val = model.predict(val_inp, batch_size=10000,verbose=0)[:,1].reshape(-1,1)
            ls = loss_beta_cox(beta_hat,Z_val, g_val, et_val, st_val, E_val)

            all_bets += [beta_hat]
            all_scores += [ls]
            if  ls < best_val:
                best_val = ls
                best_beta = beta_hat
                best_conif = nn_config.copy()
            
    bets_cox += [best_beta]
    all_bets_all_cox += [all_bets]
    all_scores_all_cox += [all_scores]
    print("###################")

β_hat cox: [[ 0.1153995 ]
 [-0.28992057]
 [-0.02270232]
 [-0.08830107]]
[-0.05293414 -0.43626466 -0.18940894 -0.18685225]
[-0.26699912 -0.59836565 -0.29287004 -0.25755893]
!!!!!
[ 0.01955872 -0.39534274 -0.10628603 -0.15927552]
[-0.20442711 -0.58486842 -0.22193157 -0.30009269]
!!!!!
[ 0.00905921 -0.4047682  -0.12827829 -0.17112143]
[-0.23093103 -0.61403411 -0.2594599  -0.25367758]
!!!!!
[ 0.04876241 -0.3724203  -0.10485393 -0.1460787 ]
[-0.162004   -0.60897822 -0.23589584 -0.26205327]
!!!!!
[-0.0366934  -0.4392166  -0.16476162 -0.19104387]
[-0.24877268 -0.61670831 -0.27388006 -0.27829065]
!!!!!
[ 0.05093508 -0.3657535  -0.10962017 -0.12956576]
[-0.19405456 -0.59098497 -0.25005857 -0.2334586 ]
!!!!!
[-0.02097394 -0.4171991  -0.14900632 -0.15926576]
[-0.23276352 -0.58693244 -0.29588529 -0.24073834]
!!!!!
[ 0.04984957 -0.3746387  -0.12258635 -0.14410245]
[-0.18701756 -0.53236045 -0.26517109 -0.23969818]
!!!!!
[-0.02942516 -0.4201266  -0.15030023 -0.17724901]
[-0.25035452 -0.59945878 -0.27

In [40]:
bets_cox

[array([-0.18362812, -0.54837559, -0.24166648, -0.25069118]),
 array([-0.17213766, -0.57335352, -0.24116706, -0.22134101]),
 array([-0.18493533, -0.50990546, -0.17833066, -0.3213099 ]),
 array([-0.17681084, -0.48531101, -0.20896033, -0.26710497]),
 array([-0.22213746, -0.45728376, -0.2018379 , -0.20673056])]

In [41]:
k = 13
for i in range(5):
    aa = np.stack(all_bets_all_cox[i])
    bb = all_scores_all_cox[i]
    print(np.nanmean(aa[np.argsort(bb)[:k],:],axis=0))

[-0.19930444 -0.58463616 -0.24681687 -0.25403143]
[-0.12110355 -0.5476493  -0.21845007 -0.18887724]
[-0.1617608  -0.47258564 -0.17380807 -0.30656474]
[-0.18076095 -0.47891231 -0.22029842 -0.2645052 ]
[-0.22161641 -0.47455383 -0.22353451 -0.18004274]


In [42]:
k = 5
for i in range(5):
    aa = np.stack(all_bets_all_cox[i])
    bb = all_scores_all_cox[i]
    print(np.nanmean(aa[np.argsort(bb)[:k],:],axis=0))

[-0.20026744 -0.58616147 -0.24267795 -0.26705541]
[-0.16528818 -0.56492081 -0.23778268 -0.20678663]
[-0.15884851 -0.47240726 -0.16357968 -0.29496776]
[-0.1745133  -0.47911361 -0.20915251 -0.26685098]
[-0.20636226 -0.48487911 -0.21232001 -0.18943906]


In [43]:
def build_mse_separate(input_dim_X, nn_config):
    """
    Build g(X,t) = g_X(X) + g_t(t) with two independent MLPs and an additive head.

    nn_config keys (with fallbacks):
      - activation: str (e.g., "relu")
      - dropout: float in [0,1]
      - optimizer: str or optimizer instance (e.g., "adam")
      - learning_rate: float
      - # X-branch
      - n_hidden_layers_x: int
      - hidden_layers_nodes_x: int
      - # t-branch
      - n_hidden_layers_t: int
      - hidden_layers_nodes_t: int
      - # (optional) output_bias_init: float or None
    """
    act   = nn_config.get("activation", "relu")
    dr    = nn_config.get("dropout", 0.0)

    # allow separate sizes/depths per branch; fallback to shared defaults if missing
    nLx   = nn_config.get("n_hidden_layers_x", nn_config.get("n_hidden_layers", 2))
    Wx    = nn_config.get("hidden_layers_nodes_x", nn_config.get("hidden_layers_nodes", 64))
    nLt   = nn_config.get("n_hidden_layers_t", nn_config.get("n_hidden_layers", 2))
    Wt    = nn_config.get("hidden_layers_nodes_t", nn_config.get("hidden_layers_nodes", 64))

    # Inputs
    X_input  = Input(shape=(input_dim_X,), name="X_input")
    t_input  = Input(shape=(1,),            name="t_input")

    # ---- g_X(X) branch ----
    x = X_input
    for _ in range(nLx):
        x = Dense(Wx, activation=act)(x)
        if dr and dr > 0:
            x = Dropout(dr)(x)
    gX = Dense(1, activation="linear", name="gX_out")(x)

    # ---- g_t(t) branch ----
    tt = t_input
    for _ in range(nLt):
        tt = Dense(Wt, activation=act)(tt)
        if dr and dr > 0:
            tt = Dropout(dr)(tt)
    gt = Dense(1, activation="linear", name="gt_out")(tt)

    # ---- Additive head: g(X,t) = g_X(X) + g_t(t) ----
    g_out = Add(name="g_out")([gX, gt])

    model = Model(inputs=[X_input, t_input], outputs=g_out, name="Additive_gX_plus_gt")

    # Optimizer
    optimizer = tf.keras.optimizers.get(nn_config.get("optimizer", "adam"))
    if hasattr(optimizer, "learning_rate"):
        optimizer.learning_rate = nn_config.get("learning_rate", 1e-3)

    model.compile(optimizer=optimizer, loss="mse", jit_compile=False)
    return model




In [44]:
from sklearn.model_selection import GroupKFold
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

def crossfit_event_residuals_seperate(df, nn_config, K=5, time_scale=30.0):
    """
    OOF residuals at EVENTS ONLY for correlation penalty.
    Trains each r_j on train-fold EVENTS and predicts for held-out fold EVENTS.
    Returns r_oof with shape (n, p), filled on events (delta==1), zeros elsewhere.
    """

    X_cols = [c for c in df.columns if c.startswith("X")]
    Z_cols = [c for c in df.columns if c.startswith("Z")]
    ids = df["id"]
    
    X = df[X_cols].values
    Z = df[Z_cols].values
    et = df["et"].values.reshape(-1, 1)
    st = df["st"].values.reshape(-1, 1)
    E = df["E"].values.reshape(-1, 1)

    X = np.asarray(X, dtype=np.float32)
    Z = np.asarray(Z, dtype=np.float32)
    et = np.asarray(et, dtype=np.float32).reshape(-1, 1)
    E = np.asarray(E, dtype=np.int32).reshape(-1)
    ids = np.asarray(ids)

    n, p = Z.shape
    r_oof = np.zeros((n, p), dtype=np.float32)

    # split by subject to avoid leakage across the same id
    gkf = GroupKFold(n_splits=min(K, np.unique(ids).size))
    for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, groups=ids)):
        tr_ev = tr_idx[E[tr_idx] == 1]
        te_ev = te_idx[E[te_idx] == 1]
        if tr_ev.size == 0 or te_ev.size == 0:
            continue  # skip degenerate fold

        for j in range(p):
            rmodel = build_mse_separate(input_dim_X=X.shape[1], nn_config=nn_config)
            es = EarlyStopping(monitor='val_loss', patience=nn_config["patience"],
                               restore_best_weights=True)
            # fit only on EVENTS in the training folds
            rmodel.fit(
                [X[tr_ev], et[tr_ev] / time_scale], Z[tr_ev, j],
                validation_data=([X[te_ev], et[te_ev] / time_scale], Z[te_ev, j]),
                batch_size=nn_config["batch_size"],
                epochs=5000, callbacks=[es], verbose=0
            )
            # predict on EVENTS in the held-out fold
            pred_te_ev = rmodel.predict([X[te_ev], et[te_ev] / time_scale], verbose=0,batch_size=10000).reshape(-1)
            r_oof[te_ev, j] = Z[te_ev, j] - pred_te_ev
        tf.keras.backend.clear_session()
        gc.collect()   
    # non-events remain zero; OK because penalty uses event weights
    return r_oof


In [45]:
r_oof = crossfit_event_residuals_seperate(df_all,nn_config)

In [46]:
np.diag(np.linalg.inv(r_oof.T @ r_oof))**0.5

array([0.11307208, 0.10808653, 0.11953788, 0.12394428], dtype=float32)